In [1]:
# 02_feature_engineering.ipynb

# 📦 Imports
import pandas as pd
import numpy as np



In [2]:
# 🔧 Config

# Leagues:
# Premier-League - 9
# La-Liga - 12
# Serie-A - 11
# Ligue-1
# Bundesliga - 20

league = 'Premier-League'
season = "2024-2025"  # change to 2024-2025 when needed
filename = f"ags_consolidated_{league}_{season}.csv"
rolling_window = 3  # number of previous matches to use in rolling averages
min_gameweek_for_features = 4  # earliest week with reliable rolling features
export = True
processed_data_path = '/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/anytime_goalscorer/data/processed'

In [3]:
filename = f"{processed_data_path}/ags_consolidated_{league}_{season}.csv"
final_df = pd.read_csv(filename)
final_df = final_df.drop(columns=[col for col in final_df.columns if "Unnamed" in col])


In [4]:
# 🗓️ Date and Gameweek
final_df["match_date"] = pd.to_datetime(final_df["match_date"])
final_df = final_df.sort_values(["team", "player", "match_date"]).reset_index(drop=True)
final_df["gameweek_number"] = final_df.groupby("team")["match_date"].rank(method="dense").astype(int)

In [8]:
player_match_unique = final_df.copy()
player_match_unique = player_match_unique.drop(columns=['bookmaker','ags_odds']).drop_duplicates()

player_match_lines = final_df.copy()
player_match_lines = player_match_lines[['match_date','player','bookmaker','ags_odds']]



In [10]:
# ✅ Rolling per90 stats based on rolling totals
def safe_per90(numerator, minutes):
    return numerator / (minutes / 90 + 1e-6)

# ⚙️ Setup
rolling_window = 3  # adjust as needed

player_match_unique = final_df.copy()
player_match_unique = player_match_unique.drop(columns=['bookmaker', 'ags_odds']).drop_duplicates()

player_match_lines = final_df.copy()
player_match_lines = final_df[['match_date', 'player', 'bookmaker', 'ags_odds']]

# ⚽ Rolling totals for raw stats and minutes
raw_fields = [
    "shots_on_target", "goals", "shots", "xg", "npxg", "xg_assist", "sca", "gca",
    "touches_att_3rd", "touches_att_pen_area", "miscontrols", "dispossessed",
    "passes_received", "progressive_passes_received", "take_ons_won", "take_ons_tackled", "minutes"
]

for col in raw_fields:
    player_match_unique[f"{col}_ma"] = (
        player_match_unique.groupby("player")[col]
        .transform(lambda x: x.shift().rolling(rolling_window, min_periods=1).mean())
    )

# ⚖️ Per 90 stats
player_match_unique["goals_per90_ma"] = safe_per90(player_match_unique["goals_ma"], player_match_unique["minutes_ma"])
player_match_unique["sot_per90_ma"] = safe_per90(player_match_unique["shots_on_target_ma"], player_match_unique["minutes_ma"])
player_match_unique["shots_per90_ma"] = safe_per90(player_match_unique["shots_ma"], player_match_unique["minutes_ma"])
player_match_unique["xg_per90_ma"] = safe_per90(player_match_unique["xg_ma"], player_match_unique["minutes_ma"])
player_match_unique["npxg_per90_ma"] = safe_per90(player_match_unique["npxg_ma"], player_match_unique["minutes_ma"])
player_match_unique["xa_per90_ma"] = safe_per90(player_match_unique["xg_assist_ma"], player_match_unique["minutes_ma"])
player_match_unique["sca_per90_ma"] = safe_per90(player_match_unique["sca_ma"], player_match_unique["minutes_ma"])
player_match_unique["gca_per90_ma"] = safe_per90(player_match_unique["gca_ma"], player_match_unique["minutes_ma"])
player_match_unique["touches_att_third_per90_ma"] = safe_per90(player_match_unique["touches_att_3rd_ma"], player_match_unique["minutes_ma"])
player_match_unique["touches_att_pen_area_per90_ma"] = safe_per90(player_match_unique["touches_att_pen_area_ma"], player_match_unique["minutes_ma"])
player_match_unique["miscontrols_per90_ma"] = safe_per90(player_match_unique["miscontrols_ma"], player_match_unique["minutes_ma"])
player_match_unique["dispossessed_per90_ma"] = safe_per90(player_match_unique["dispossessed_ma"], player_match_unique["minutes_ma"])
player_match_unique["passes_received_per90_ma"] = safe_per90(player_match_unique["passes_received_ma"], player_match_unique["minutes_ma"])
player_match_unique["progressive_passes_received_per90_ma"] = safe_per90(player_match_unique["progressive_passes_received_ma"], player_match_unique["minutes_ma"])
player_match_unique["take_ons_won_per90_ma"] = safe_per90(player_match_unique["take_ons_won_ma"], player_match_unique["minutes_ma"])
player_match_unique["take_ons_tackled_per90_ma"] = safe_per90(player_match_unique["take_ons_tackled_ma"], player_match_unique["minutes_ma"])

# 📊 Team-Level Rolling Stats
player_match_unique = player_match_unique.sort_values(["team", "match_date"])
player_match_unique["team_shots_ma"] = player_match_unique.groupby("team")["team_shots"].transform(lambda x: x.shift().rolling(rolling_window, min_periods=1).mean())
player_match_unique["team_shots_on_target_ma"] = player_match_unique.groupby("team")["team_shots_on_target"].transform(lambda x: x.shift().rolling(rolling_window, min_periods=1).mean())
player_match_unique["team_xG_ma"] = player_match_unique.groupby("team")["team_xG"].transform(lambda x: x.shift().rolling(rolling_window, min_periods=1).mean())

# 📈 Share Features
player_match_unique["sot_team_share"] = player_match_unique["sot_per90_ma"] / (player_match_unique["team_shots_on_target_ma"] + 0.01)
player_match_unique["shots_team_share"] = player_match_unique["shots_per90_ma"] / (player_match_unique["team_shots_ma"] + 0.01)
player_match_unique["xg_team_share"] = player_match_unique["xg_per90_ma"] / (player_match_unique["team_xG_ma"] + 0.01)

# 🧬 Merge enriched features back
final_df = pd.merge(player_match_lines, player_match_unique, how="inner", on=["match_date", "player"])

# 📉 Market Sentiment
final_df["implied_prob_ags"] = 1 / final_df["ags_odds"]

# ⚙️ Contextual Features
final_df["is_home"] = (final_df["team"] == final_df["home_team"]).astype(int)
final_df["opponent_sot_allowed_per90"] = final_df["opp_shots_on_target"] / 90
final_df["opponent_xg_allowed_per90"] = final_df["opp_xG"] / 90

# ✅ Add is_penalty_taker field
penalty_totals = final_df.groupby(["team", "player"])["pens_att"].sum().reset_index()
penalty_takers = (
    penalty_totals.sort_values(["team", "pens_att"], ascending=[True, False])
    .groupby("team")
    .first()
    .reset_index()
    .rename(columns={"player": "penalty_taker"})
)
final_df = final_df.merge(penalty_takers[["team", "penalty_taker"]], on="team", how="left")
final_df["is_penalty_taker"] = (final_df["player"] == final_df["penalty_taker"]).astype(int)
final_df = final_df.drop(columns=["penalty_taker"])


In [11]:
# 🚦 Model Confidence (simple GW-based logic for now)
final_df = final_df[final_df["gameweek_number"] >= min_gameweek_for_features].copy()
final_df["model_confidence"] = np.where(
    final_df["gameweek_number"] <= min_gameweek_for_features + 1, "low", "high"
)


In [12]:
# 🎯 Target Variable
final_df["target_hit_line"] = (final_df["goals"] >= 1).astype(int)




In [13]:
final_df.columns

Index(['match_date', 'player', 'bookmaker', 'ags_odds', 'team', 'position',
       'age_decimal', 'minutes', 'goals', 'shots', 'shots_on_target', 'xg',
       'npxg', 'xg_assist', 'sca', 'gca', 'pens_att', 'miscontrols',
       'dispossessed', 'passes_received', 'progressive_passes_received',
       'take_ons', 'take_ons_won', 'take_ons_tackled', 'touches_att_3rd',
       'touches_att_pen_area', 'team_shots', 'team_shots_on_target', 'team_xG',
       'home_team', 'away_team', 'opp', 'opp_xG', 'team_score', 'opp_score',
       'team_possession', 'opp_possession', 'team_passing_accuracy',
       'opp_passing_accuracy', 'opp_shots_on_target', 'opp_shots',
       'team_saves', 'opp_saves', 'team_saves_faced', 'opp_saves_faced',
       'gameweek_number', 'shots_on_target_ma', 'goals_ma', 'shots_ma',
       'xg_ma', 'npxg_ma', 'xg_assist_ma', 'sca_ma', 'gca_ma',
       'touches_att_3rd_ma', 'touches_att_pen_area_ma', 'miscontrols_ma',
       'dispossessed_ma', 'passes_received_ma',
       'p

In [14]:
if export:
    export_filename = f"{processed_data_path}/features_{league}_{season}.csv"
    final_df.to_csv(export_filename, index=False)

print("✅ Feature engineering complete.")


✅ Feature engineering complete.
